# 01 · Intact pipe against the closed-form T(0,1) solution

**Shows.** The explicit solver on an intact pipe, its convergence, and its cost against
COMSOL's explicit solver at the same accuracy.

**Problem.** Steel pipe, outer radius 30 mm, wall 4 mm, half model with antisymmetry on
y = 0. Torsional traction on the end z = L, proportional to r, 30 kHz 5-cycle Hann burst,
1 MPa. t_max = 400 us × L / 0.5 m, so the reflections from both ends are included.
Sensors on the outer surface at z = L/5, L/2, 4L/5.

**Validated against.** The closed form `analytic_surface_velocity`: T(0,1) is
non-dispersive, so v_θ = τ0 / (ρ c_T) f(t − (L − z)/c_T), plus the end reflections by
the method of images. Error: NRMSE of v_θ in % of the range of the exact signal.

**Run time.** Section 1 solves a small case here (about 30 s with 8 threads). Sections 2
and 3 read `results/intact_pipe.csv`, produced by

```
julia -t 8 --project=. scripts/run_intact_pipe.jl directions   # ~15 min
julia -t 8 --project=. scripts/run_intact_pipe.jl length       # ~80 min
```

In [ ]:
using Pkg
Pkg.activate(joinpath(@__DIR__, ".."))   # notebooks/Project.toml
using ExplicitGuidedWaves
using Plots, CSV, DataFrames, Printf

const PROJECT_DIR = pkgdir(ExplicitGuidedWaves)
const RESULTS_DIR = joinpath(PROJECT_DIR, "results")
const DATA_DIR = joinpath(PROJECT_DIR, "data")
const FIGURES_DIR = joinpath(RESULTS_DIR, "figures")
mkpath(FIGURES_DIR)
gr(fmt = :png)
println("Julia threads: ", Threads.nthreads())

In [ ]:
"Read a results table of `results_dir`, or stop with the command that produces it."
function results_table(results_dir, file, command)
    path = joinpath(results_dir, file)
    if !isfile(path)
        error("$file not found. From the project folder run:\n  $command")
    end
    return DataFrame(CSV.File(path))
end

## 1. A small case, solved here

L = 0.5 m, order 2, one cell through the wall, 24 sectors on the half circumference,
5 mm along the pipe (88 641 degrees of freedom).

In [ ]:
warm_up_solver()
small_case = IntactPipeCase(0.5; order = 2, n_circumferential = 24, n_axial = 100)
small_result = run_intact_pipe(small_case)
@printf("NRMSE of v_theta at L/5, L/2, 4L/5: %.3e / %.3e / %.3e %%\n", small_result.nrmse_percent...)

In [ ]:
time_us = 1e6 .* small_result.times
sensor_column = 2   # z = L/2
signal_plot = plot(time_us, small_result.exact[:, sensor_column]; label = "closed form",
                   linewidth = 2, color = MODEL_COLORS.closed_form, ylabel = "v_θ (m/s)")
plot!(signal_plot, time_us, small_result.numerical[:, sensor_column]; label = "explicit FEM",
      linestyle = :dash, color = MODEL_COLORS.explicit_fem)
difference = small_result.numerical[:, sensor_column] .- small_result.exact[:, sensor_column]
error_plot = plot(time_us, difference; label = "", color = MODEL_COLORS.explicit_fem, xlabel = "t (μs)",
                  ylabel = "difference (m/s)")
figure = plot(signal_plot, error_plot; layout = (2, 1), size = (700, 500))
savefig(figure, joinpath(FIGURES_DIR, "intact_small_case_z_L_over_2.png"))
figure

## 2. Which direction limits the accuracy

Series A refines the circumference with the axis fixed at 5 mm; series B refines the
axis with 48 sectors. If A drops and B does not, the circumference (the polygonal
cross-section) limits the accuracy.

In [ ]:
intact = results_table(RESULTS_DIR, "intact_pipe.csv", "julia -t 8 --project=. scripts/run_intact_pipe.jl directions")
direction_cases = Set(case_name.(INTACT_SERIES["directions"]))
directions = filter(row -> row.case in direction_cases, intact)
directions.h_axial_mm = 1000 .* directions.length_m ./ directions.n_axial
# series A: 100 axial cells (5 mm), sectors refined; series B: the rest (48 sectors, axis refined).
# The case 48 x 100 belongs to both; it is listed in A.
directions.series = [cells == 100 ? "A" : "B" for cells in directions.n_axial]
sort!(directions, [:series, :n_circumferential, :n_axial])
select(directions, :series, :n_circumferential, :n_axial, :h_axial_mm, :n_dofs,
       :nrmse_1_percent, :nrmse_2_percent, :nrmse_3_percent)

## 3. Cost against COMSOL at matched accuracy, and against pipe length

COMSOL: Elastic Waves, Time Explicit (quartic nodal DG), hx = 20 mm, 8 threads, same
laptop (`data/reference_costs.csv`).

In [ ]:
length_cases = Set(case_name.(INTACT_SERIES["length"]))
length_series = sort(filter(row -> row.case in length_cases, intact), :length_m)
comsol = filter(row -> row.case == "intact_hx20mm",
                DataFrame(CSV.File(joinpath(DATA_DIR, "reference_costs.csv"))))
cost = innerjoin(select(length_series, :length_m, :n_dofs, :n_steps, :loop_s, :threads,
                        :nrmse_1_percent, :nrmse_3_percent),
                 select(comsol, :length_m, :time_s => :comsol_s, :nrmse_percent => :comsol_nrmse);
                 on = :length_m)
cost.comsol_over_fem = round.(cost.comsol_s ./ cost.loop_s; digits = 1)
CSV.write(joinpath(RESULTS_DIR, "cost_against_comsol.csv"), cost)
cost

In [ ]:
growth = log.(length_series.loop_s[2:end] ./ length_series.loop_s[1:end-1]) ./
         log.(length_series.length_m[2:end] ./ length_series.length_m[1:end-1])
@printf("loop time grows as L^%.2f between successive lengths\n", sum(growth) / length(growth))

**Reading the result.** At L = 0.5 m this code reaches a smaller error than COMSOL's
explicit solver with hx = 20 mm, in less time with the same threads (column
`comsol_over_fem`): most of COMSOL's explicit cost is its implementation (quartic DG), not
the explicit method. The loop time grows as about L^2, because both the degrees of freedom
and the number of steps grow with L.

The times depend on the state of the computer: on 22 Nov 2026 the same runs took
202 / 868 / 3736 s in the morning and 352 / 1359 / 5707 s in the evening (VALIDATION.md,
run-time variability), so the ratio against COMSOL is 7-12. Compare times measured in one
session, run back to back.